# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5.

In [1]:
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

{'A': 2, 'B': 1}


In [2]:
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

[0, 1, 1]


In [3]:
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

2


In [4]:
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

{'ab': 2, 'abc': 3}


In [5]:
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

5
None


In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [6]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [8]:
# База
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")
#1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
def to_num(n):
    return float(n.strip().replace(" ", ""))
def clean_city(n):
    city = n.strip().capitalize()
    match city:
        case "Алматы" | "Almaty":
            return "Almaty"
        case "Астана" | "Astana":
            return "Astana"
        case "Шымкент" | "Shymkent":
            return "Shymkent"
        case "Караганда" | "Karaganda":
            return "Karaganda"
        case "Актобе" | "Aktobe":
            return "Aktobe"
        case "Атырау" | "Atyrau":
            return "Atyrau"
        case _:
            raise ValueError(f"Неизвестный город: {n}")

for item in rows:
    item["city"]=clean_city(item["city"])
    item["amount"]=to_num(item["amount"])

5463 строк


In [7]:
#2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
city_dict = {};
order_by_cat_dict = {};
for item in rows:
    order_by_cat_dict[item["category"]]= order_by_cat_dict.get(item["category"],0) + 1
    city_dict[item["city"]]= city_dict.get(item["city"],0) + 1
print(order_by_cat_dict)
for c, v in sorted(order_by_cat_dict.items(), key=lambda t: t[1], reverse=True):
    print(f"{c:<10} {v:>7}")

{'Одежда': 1529, 'Красота': 991, 'Дом': 1105, 'Книги': 653, 'Электроника': 1185}
Одежда        1529
Электроника    1185
Дом           1105
Красота        991
Книги          653


In [48]:
#3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
order_by_cat_dict = {};
for item in rows:
    if item["status"]=="оплачен":
        order_by_cat_dict.setdefault(item["category"], []).append(item["amount"])
# print(order_by_cat_dict)
for c, v in sorted(order_by_cat_dict.items(), key=lambda t:sum(t[1]), reverse=True):
    print(f"{c:<12}| кол-во {len(v):>6,}| Итого {sum(v):>12,} ₸| средний {round(sum(v)/len(v),2):>12,} ₸")

Электроника | кол-во  1,065| Итого 65,270,240.0 ₸| средний    61,286.61 ₸
Одежда      | кол-во  1,379| Итого 18,321,080.0 ₸| средний    13,285.77 ₸
Дом         | кол-во    987| Итого 10,516,320.0 ₸| средний    10,654.83 ₸
Красота     | кол-во    890| Итого  5,507,620.0 ₸| средний     6,188.34 ₸
Книги       | кол-во    583| Итого  2,294,100.0 ₸| средний     3,934.99 ₸


In [53]:
#4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.
order_by_month_dict = {};
for item in rows:
    # if item["status"]=="оплачен":
    order_by_month_dict.setdefault(item["order_date"][:7], []).append(item["amount"])
# print(order_by_month_dict)
for c, v in sorted(order_by_month_dict.items()):
    print(f"{c:<12}| кол-во {len(v):>6,}| Итого {sum(v):>12,} ₸")

2026-09     | кол-во  2,536| Итого 57,489,850.0 ₸
2026-10     | кол-во  2,368| Итого 44,419,510.0 ₸


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [65]:
# Стандарт
#5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
order_by_month_dict = {};
for item in rows:
    if item["status"]=="оплачен":
        order_by_month_dict.setdefault(item["order_date"][:7], []).append(item["amount"])
# print(order_by_month_dict)

sorted_months = sorted(order_by_month_dict.items())
# for c, v in sorted_months:
#     print(f"{c:<12}| кол-во {len(v):>6,}| Итого {sum(v):>12,} ₸")

prev_month_name = None
prev_revenue = 0

for month_name, current_revenue in sorted_months:
    if prev_month_name is None:
        change_str = "---"
    else:
        if prev_revenue > 0:
            pct_change = ((sum(current_revenue) - prev_revenue) / prev_revenue) * 100
            change_str = f"{pct_change:+.2f}%"
        else:
            change_str = "N/A (в пред. месяце 0)"

    print(f"{month_name:<10} | {sum(current_revenue):>10,} ₸ | {change_str}")

    prev_month_name = month_name
    prev_revenue = sum(current_revenue)

2026-09    | 57,489,850.0 ₸ | ---
2026-10    | 44,419,510.0 ₸ | -22.74%


In [83]:
# 6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
city_dict = {};
for item in rows:
    city = item["city"]
    status = item["status"]
    amount = item["amount"]

    city_dict.setdefault(city, {})

    city_dict[city].setdefault(status, [])

    city_dict[city][status].append(amount)

for city_value in city_dict:
    city_sum_total=0
    for status_value in city_dict[city_value]:
        city_sum_total+=sum(city_dict[city_value][status_value])
    for status_value in city_dict[city_value]:
        status_value_one=sum(city_dict[city_value][status_value])
        print(f"{city_value:<10} | {status_value:>16} | {status_value_one:>16,} ₸ | {round((status_value_one*100)/city_sum_total,2):>10} %")

Almaty     |          оплачен |     60,166,930.0 ₸ |      96.02 %
Almaty     |          отменён |      2,493,740.0 ₸ |       3.98 %
Astana     |          оплачен |     18,424,300.0 ₸ |       91.7 %
Astana     |          отменён |      1,666,840.0 ₸ |        8.3 %
Karaganda  |          оплачен |      7,158,850.0 ₸ |      90.76 %
Karaganda  |          отменён |        728,890.0 ₸ |       9.24 %
Shymkent   |          отменён |      2,409,310.0 ₸ |      21.13 %
Shymkent   |          оплачен |      8,993,810.0 ₸ |      78.87 %
Atyrau     |          оплачен |      1,923,620.0 ₸ |      90.55 %
Atyrau     |          отменён |        200,860.0 ₸ |       9.45 %
Aktobe     |          оплачен |      5,241,850.0 ₸ |       91.7 %
Aktobe     |          отменён |        474,360.0 ₸ |        8.3 %


In [88]:
# 7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
customer_dict = {};
for item in rows:
    customer = item["customer_id"]
    status = item["status"]
    amount = item["amount"]
    if status=="оплачен":
        customer_dict.setdefault(customer, []).append(amount)

top=5
cc=0
for c, v in sorted(customer_dict.items(), key=lambda t:sum(t[1]), reverse=True):
    print(f"{c:<12}| кол-во {len(v):>6,}| Итого {sum(v):>12,} ₸| средний {round(sum(v)/len(v),2):>12,} ₸")
    cc+=1
    if top==cc:
        break


TEST        | кол-во      3| Итого 29,999,970.0 ₸| средний  9,999,990.0 ₸
C01627      | кол-во      7| Итого    257,180.0 ₸| средний     36,740.0 ₸
C00485      | кол-во      6| Итого    215,920.0 ₸| средний    35,986.67 ₸
C01338      | кол-во      6| Итого    192,080.0 ₸| средний    32,013.33 ₸
C02122      | кол-во      5| Итого    191,670.0 ₸| средний     38,334.0 ₸


Пользователь тест имеет отладочный характер

In [95]:
# 8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.
#
customer_by_city_dict = {};
for item in rows:
    city = item["city"]
    customer = item["customer_id"]

    amount = item["amount"]
    if status=="оплачен":
        customer_by_city_dict.setdefault(city, [])
        customer_by_city_dict[city].append(customer)

for city in customer_by_city_dict:
    unique_list = list(set(customer_by_city_dict[city]))
    unique_count = len(unique_list)
    print(f"{city:<12}| общее кол-во {len(customer_by_city_dict[city]):>6,}| кол-во {unique_count:>6,}")


Almaty      | общее кол-во  2,188| кол-во  1,351
Astana      | общее кол-во  1,390| кол-во    981
Karaganda   | общее кол-во    538| кол-во    451
Shymkent    | общее кол-во    804| кол-во    655
Atyrau      | общее кол-во    146| кол-во    138
Aktobe      | общее кол-во    397| кол-во    358


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [9]:
# Челлендж
# Найдите полные дубли строк: превратите каждую строку в tuple(r.values()) и проверяйте через множество. Сколько их и в каком месяце?
order_by_month_dict = {};
for item in rows:
    order_by_month_dict.setdefault(item["order_date"][:7], []).append(item)
# print(order_by_month_dict)

sorted_months = sorted(order_by_month_dict.items())
for month_name, current_revenue in sorted_months:
    unique_rows = []
    seen_tuples = set()
    for r in current_revenue:
        row_tuple = tuple(r.values())
        if row_tuple not in seen_tuples:
            seen_tuples.add(row_tuple)
            unique_rows.append(r)

    print(f"{month_name:<12}| общее кол-во {len(current_revenue):>6,}| кол-во {len(unique_rows):>6,}")


2026-09     | общее кол-во  2,782| кол-во  2,602
2026-10     | общее кол-во  2,681| кол-во  2,681


In [25]:
# Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
order_by_month_dict = {};
order_by_month_dict_uniq = {};
for item in rows:
    order_by_month_dict.setdefault(item["order_date"][:7], []).append(item)
# print(order_by_month_dict)

sorted_months = sorted(order_by_month_dict.items())
for month_name, current_revenue in sorted_months:

    unique_rows = []
    seen_tuples = set()
    for r in current_revenue:
        if r["customer_id"]=="TEST":
            continue
        if r["status"]!="оплачен":
            continue
        row_tuple = tuple(r.values())
        if row_tuple not in seen_tuples:
            seen_tuples.add(row_tuple)
            unique_rows.append(r)
    order_by_month_dict_uniq.setdefault(month_name, unique_rows)

    print(f"{month_name:<12}| общее кол-во {len(current_revenue):>6,}| кол-во {len(unique_rows):>6,}")
print("---"*20)
prev_month_name = None
prev_revenue = 0
#current_revenue
for month_name in order_by_month_dict_uniq:
    lsum=sum(r["amount"] for r in order_by_month_dict_uniq[month_name])
    if prev_month_name is None:
        change_str = "---"
    else:
        if prev_revenue > 0:
            pct_change = ((lsum - prev_revenue) / prev_revenue) * 100
            change_str = f"{pct_change:+.2f}%"
        else:
            change_str = "N/A (в пред. месяце 0)"

    print(f"{month_name:<10} | {lsum:>10,} ₸ | {change_str}")

    prev_month_name = month_name
    prev_revenue = lsum

2026-09     | общее кол-во  2,782| кол-во  2,372
2026-10     | общее кол-во  2,681| кол-во  2,367
------------------------------------------------------------
2026-09    | 35,243,410.0 ₸ | ---
2026-10    | 34,419,520.0 ₸ | -2.34%


In [36]:
# У какого клиента больше всего заказов (после удаления дублей)? Сколько?
unique_rows = []
only_paid_rows = []
seen_tuples = set()
for item in rows:
    customer = item["customer_id"]
    status = item["status"]
    amount = item["amount"]
    if customer=="TEST":
        continue
    if status!="оплачен":
        continue
    only_paid_rows.append(item)

    row_tuple = tuple(item.values())
    if row_tuple not in seen_tuples:
        seen_tuples.add(row_tuple)
        unique_rows.append(item)
print("---"*20)
print(f" общее кол-во {len(rows):>6,}| Итого {sum(r["amount"] for r in rows):>6,} ₸")
print("---"*20)
print("-Оплаченных-")
print(f" общее кол-во {len(only_paid_rows):>6,}| Итого {sum(r["amount"] for r in only_paid_rows):>6,} ₸")
print("---"*20)
print("-Уникальных оплаченных-")
print(f" общее кол-во {len(unique_rows):>6,}| Итого {sum(r["amount"] for r in unique_rows):>6,} ₸")
print("---"*20)
customer_dict = {};
for sss in unique_rows:
    customer = sss["customer_id"]
    status = sss["status"]
    amount = sss["amount"]
    customer_dict.setdefault(customer, []).append(amount)
# print("---"*20)
top=5
cc=0
for c, v in sorted(customer_dict.items(), key=lambda t:sum(t[1]), reverse=True):
    print(f"{c:<12}| кол-во {len(v):>6,}| Итого {sum(v):>12,} ₸| средний {round(sum(v)/len(v),2):>12,} ₸")
    cc+=1
    if top==cc:
        break
# Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.


------------------------------------------------------------
 общее кол-во  5,463| Итого 109,883,360.0 ₸
------------------------------------------------------------
-Оплаченных-
 общее кол-во  4,901| Итого 71,909,390.0 ₸
------------------------------------------------------------
-Уникальных оплаченных-
 общее кол-во  4,739| Итого 69,662,930.0 ₸
------------------------------------------------------------
C00485      | кол-во      6| Итого    215,920.0 ₸| средний    35,986.67 ₸
C02122      | кол-во      5| Итого    191,670.0 ₸| средний     38,334.0 ₸
C00400      | кол-во      6| Итого    191,550.0 ₸| средний     31,925.0 ₸
C00048      | кол-во      6| Итого    176,740.0 ₸| средний    29,456.67 ₸
C00588      | кол-во      6| Итого    175,090.0 ₸| средний    29,181.67 ₸
